# Compare MoE vs MLP Across All Metrics

This notebook compares one MoE checkpoint against one MLP checkpoint on the same test set, using the same evaluator pipeline as the project code.

It reports:
- full evaluator metrics side by side
- absolute and relative differences
- per-instance win rates for objective, merit, and violations

In [ ]:
import os
import glob
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

from utils.trainer import load_instance, DEVICE
from utils.evaluator import Evaluator
from eval import load_model_from_checkpoint, resolve_checkpoints

print(f'Device: {DEVICE}')

## 1) Configure Paths
Set either `*_RUN_DIR` or `*_CKPT_PATH` for each model.

In [ ]:
# Optional run directories (auto-resolve model.pt or members/member_*.pt)
MLP_RUN_DIR = None
MOE_RUN_DIR = None

# Optional explicit checkpoint paths
MLP_CKPT_PATH = None
MOE_CKPT_PATH = None

# Example:
# MLP_RUN_DIR = 'results/nonsmooth_nonconvex/socp/..._FSNet_seed0_e300_lr1e-04_n7000'
# MOE_RUN_DIR = 'results/nonsmooth_nonconvex/socp/..._FSNet_seed0_e300_lr5e-05_n7000'

BATCH_SIZE = 512
FORCE_EVAL_MODE = True  # ensure post-processing is active for FSNet/S3Net

In [ ]:
def resolve_single_checkpoint(run_dir, ckpt_path, label):
    if ckpt_path:
        if not os.path.isfile(ckpt_path):
            raise FileNotFoundError(f'{label} checkpoint not found: {ckpt_path}')
        return ckpt_path

    if run_dir:
        ckpts = resolve_checkpoints(run_dir)
        if len(ckpts) != 1:
            raise ValueError(
                f'{label}: expected exactly 1 checkpoint from run dir, got {len(ckpts)}. '
                'Set an explicit *_CKPT_PATH if this is an ensemble run.'
            )
        return ckpts[0]

    raise ValueError(f'Set either {label}_RUN_DIR or {label}_CKPT_PATH')


def safe_ratio_pct(delta, baseline):
    denom = abs(baseline) if abs(baseline) > 1e-12 else np.nan
    return 100.0 * delta / denom


def collect_per_instance_metrics(model, evaluator, opt_problem, loader, penalty=1e6):
    obj_all = []
    merit_all = []
    eq_l1_all = []
    ineq_l1_all = []
    opt_gap_all = []

    with torch.no_grad():
        for x_batch, y_true in loader:
            x_batch = x_batch.to(DEVICE)
            y_true = y_true.to(DEVICE)

            y_pred = model(x_batch)
            y_scaled = opt_problem.scale(y_pred)
            with torch.enable_grad():
                y_final = evaluator._post_process_predictions(x_batch, y_scaled)

            obj = opt_problem.obj_fn(y_final)
            obj_true = opt_problem.obj_fn(y_true)
            eq_l1 = opt_problem.eq_resid(x_batch, y_final).abs().sum(dim=1)
            ineq_l1 = opt_problem.ineq_resid(x_batch, y_final).abs().sum(dim=1)
            merit = obj + penalty * (eq_l1 + ineq_l1)
            opt_gap = 100.0 * (obj - obj_true) / obj_true.abs().clamp_min(1e-12)

            obj_all.append(obj.cpu().numpy())
            merit_all.append(merit.cpu().numpy())
            eq_l1_all.append(eq_l1.cpu().numpy())
            ineq_l1_all.append(ineq_l1.cpu().numpy())
            opt_gap_all.append(opt_gap.cpu().numpy())

    return {
        'objective': np.concatenate(obj_all),
        'merit': np.concatenate(merit_all),
        'eq_violation_l1': np.concatenate(eq_l1_all),
        'ineq_violation_l1': np.concatenate(ineq_l1_all),
        'opt_gap_pct': np.concatenate(opt_gap_all),
    }

In [ ]:
mlp_ckpt = resolve_single_checkpoint(MLP_RUN_DIR, MLP_CKPT_PATH, 'MLP')
moe_ckpt = resolve_single_checkpoint(MOE_RUN_DIR, MOE_CKPT_PATH, 'MOE')

print('MLP checkpoint:', mlp_ckpt)
print('MoE checkpoint:', moe_ckpt)

mlp_raw = torch.load(mlp_ckpt, map_location='cpu', weights_only=False)
moe_raw = torch.load(moe_ckpt, map_location='cpu', weights_only=False)
mlp_cfg = copy.deepcopy(mlp_raw['config'])
moe_cfg = copy.deepcopy(moe_raw['config'])

required_same = ['prob_type', 'prob_name', 'prob_size', 'method']
for k in required_same:
    if mlp_cfg.get(k) != moe_cfg.get(k):
        raise ValueError(f'Config mismatch for {k}: MLP={mlp_cfg.get(k)} vs MoE={moe_cfg.get(k)}')

print('Common setup looks compatible.')
print('Method:', mlp_cfg['method'])
print('Problem:', mlp_cfg['prob_type'], '/', mlp_cfg['prob_name'])

In [ ]:
# Build one shared optimization problem instance
base_cfg = copy.deepcopy(mlp_cfg)
if FORCE_EVAL_MODE:
    base_cfg['_eval_only'] = True

opt_problem, _ = load_instance(base_cfg)
test_loader = DataLoader(opt_problem.test_dataset, batch_size=BATCH_SIZE, shuffle=False)

mlp_model, _ = load_model_from_checkpoint(mlp_ckpt, opt_problem)
moe_model, _ = load_model_from_checkpoint(moe_ckpt, opt_problem)

mlp_eval_cfg = copy.deepcopy(mlp_cfg)
moe_eval_cfg = copy.deepcopy(moe_cfg)
if FORCE_EVAL_MODE:
    mlp_eval_cfg['_eval_only'] = True
    moe_eval_cfg['_eval_only'] = True

mlp_evaluator = Evaluator(opt_problem, mlp_eval_cfg['method'], mlp_eval_cfg)
moe_evaluator = Evaluator(opt_problem, moe_eval_cfg['method'], moe_eval_cfg)

print(f'Test size: {len(opt_problem.test_dataset)}  | batch size: {BATCH_SIZE}')

In [ ]:
# Aggregate evaluator metrics
mlp_metrics = mlp_evaluator.evaluate(mlp_model, test_loader, split_name='MLP')
moe_metrics = moe_evaluator.evaluate(moe_model, test_loader, split_name='MoE')

all_keys = sorted(set(mlp_metrics.keys()) | set(moe_metrics.keys()))
rows = []
for k in all_keys:
    m_val = float(mlp_metrics.get(k, np.nan))
    e_val = float(moe_metrics.get(k, np.nan))
    delta = e_val - m_val
    delta_pct = safe_ratio_pct(delta, m_val)
    rows.append({
        'metric': k,
        'MLP': m_val,
        'MoE': e_val,
        'MoE_minus_MLP': delta,
        'pct_change_vs_MLP': delta_pct,
    })

cmp_df = pd.DataFrame(rows).sort_values('metric').reset_index(drop=True)
pd.set_option('display.max_rows', 500)
cmp_df

In [ ]:
# Per-instance comparison (lower is better for all listed metrics)
mlp_inst = collect_per_instance_metrics(mlp_model, mlp_evaluator, opt_problem, test_loader)
moe_inst = collect_per_instance_metrics(moe_model, moe_evaluator, opt_problem, test_loader)

summary_rows = []
for metric_name in ['objective', 'merit', 'eq_violation_l1', 'ineq_violation_l1', 'opt_gap_pct']:
    mlp_vals = mlp_inst[metric_name]
    moe_vals = moe_inst[metric_name]
    moe_better = (moe_vals < mlp_vals).mean()
    tie = (np.isclose(moe_vals, mlp_vals, rtol=1e-9, atol=1e-12)).mean()
    mlp_better = (mlp_vals < moe_vals).mean()
    summary_rows.append({
        'metric': metric_name,
        'MLP_mean': mlp_vals.mean(),
        'MoE_mean': moe_vals.mean(),
        'MoE_better_frac': moe_better,
        'Tie_frac': tie,
        'MLP_better_frac': mlp_better,
    })

inst_df = pd.DataFrame(summary_rows)
inst_df

In [ ]:
# Quick plots for core metrics
core_metrics = ['objective', 'merit_mean', 'opt_gap_mean', 'eq_violation_l1_mean', 'ineq_violation_l1_mean']
plot_df = cmp_df[cmp_df['metric'].isin(core_metrics)].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = np.arange(len(plot_df))
w = 0.38
axes[0].bar(x - w/2, plot_df['MLP'], width=w, label='MLP')
axes[0].bar(x + w/2, plot_df['MoE'], width=w, label='MoE')
axes[0].set_xticks(x)
axes[0].set_xticklabels(plot_df['metric'], rotation=35, ha='right')
axes[0].set_title('Core metrics: absolute values')
axes[0].legend()

axes[1].bar(x, plot_df['MoE_minus_MLP'])
axes[1].axhline(0.0, color='black', linewidth=1)
axes[1].set_xticks(x)
axes[1].set_xticklabels(plot_df['metric'], rotation=35, ha='right')
axes[1].set_title('MoE - MLP (negative means MoE lower)')

plt.tight_layout()
plt.show()

In [ ]:
# Optional: save tables
out_dir = 'results/moe_vs_mlp_compare'
os.makedirs(out_dir, exist_ok=True)

cmp_csv = os.path.join(out_dir, 'metrics_compare.csv')
inst_csv = os.path.join(out_dir, 'per_instance_compare.csv')
cmp_df.to_csv(cmp_csv, index=False)
inst_df.to_csv(inst_csv, index=False)

print('Saved:')
print(' -', cmp_csv)
print(' -', inst_csv)